# Attack effectiveness matrix

Thin driver for `analytics.attacks.plot_model_attack_effectiveness_matrix`. Edit config cells only; plotting logic lives in the library module.


In [1]:
import sys
from pathlib import Path

# Add src/ to sys.path before importing analytics (required for Jupyter kernels).
_project_root = Path.cwd().resolve()
while _project_root != _project_root.parent and not (_project_root / "src" / "analytics").is_dir():
    _project_root = _project_root.parent
if not (_project_root / "src" / "analytics").is_dir():
    raise FileNotFoundError(
        "Could not find the project root (expected src/analytics/). "
        "Set the notebook working directory to the QB-PIER repo root."
    )
_src = _project_root / "src"
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

from analytics.notebook_setup import notebook_paths, load_all_metrics

project_root, TRAINING_ROOT, ATTACK_EXPERIMENT_DIR, OUT, _src = notebook_paths("attack_matrix", start=_project_root)


In [5]:
from analytics.attacks import (
    ATTACK_TABLE_METRICS,
    MatrixAxisGroups,
    assign_axis_group_key,
    available_attack_metrics,
    make_metric_cell_fn,
    ordered_axis_group_labels,
    plot_model_attack_effectiveness_matrix,
)
from loading import load_attack_aggregates

# Attack-table metric (AGENTS.md §4.3)
METRIC = "ASR"  # or: flip_rate, ASR, mean_confidence_drop_all, transfer_gap, ...
GROUP_AGG = "mean"  # mean | max | auc
CELL_STYLE = "auc"  # for mean/max: auc | max_steepness | max_rel_drop

df = load_attack_aggregates(attack_experiment_dir=ATTACK_EXPERIMENT_DIR)
print(f"Loaded {len(df)} rows, {df['source_model'].nunique()} source models")
print("In dataframe:", available_attack_metrics(df))
print("Supported:", list(ATTACK_TABLE_METRICS))

CELL_FN = make_metric_cell_fn(METRIC, style=CELL_STYLE)


Loaded 48552 rows, 34 source models
In dataframe: ['robust_accuracy', 'flip_rate', 'ASR', 'mean_confidence_drop_all', 'mean_margin_drop_all', 'mean_confidence_drop_cc', 'mean_margin_drop_cc', 'transfer_gap']
Supported: ['robust_accuracy', 'flip_rate', 'ASR', 'mean_confidence_drop_all', 'mean_margin_drop_all', 'mean_confidence_drop_cc', 'mean_margin_drop_cc', 'transfer_gap']


In [6]:
# Rows involving Quantum VQM no_entangling (source or target)
_vqm_no_ent = (
    df["source_model"].str.contains("no_entangling", case=False, na=False)
    & df["source_model"].str.contains("VQM", case=False, na=False)
) | (
    df["target_model"].str.contains("no_entangling", case=False, na=False)
    & df["target_model"].str.contains("VQM", case=False, na=False)
)
df_vqm_no_entangling = df.loc[_vqm_no_ent].copy()

print(f"Selected {len(df_vqm_no_entangling)} rows (of {len(df)})")

# Remove VQM no_entangling rows from main df
df = df.loc[~_vqm_no_ent].copy()
print(f"Main df now has {len(df)} rows (removed {len(df_vqm_no_entangling)})")

Selected 2814 rows (of 48552)
Main df now has 45738 rows (removed 2814)


In [8]:
# By-architecture matrix: each arch (all variants) then best-only row/col (*).
# Best variants stay in the arch bucket; * groups sit right after their arch.
# Diagonal: white-box only (source_model == target_model), not cross-variant intra-arch.
BEST_ARCHITECTURE_VARIANTS = {
    "Classical/CNN*": ["conv_channels=[32,64], fc_sizes=[128]"],
    "Classical/MLP*": ["hidden_sizes=[1024,512,64,64]"],
    "Quantum/VQM*": ["Quantum/VQM/ansatz=strong, qdepth=200"],
}

arch_best_spec = MatrixAxisGroups(
    by="architecture",
    extra_groups=BEST_ARCHITECTURE_VARIANTS,
    diagonal_whitebox_only=True,
)

print("Axis order (rows/cols):", ordered_axis_group_labels(df["source_model"].unique(), arch_best_spec))
print("Best-variant models (also counted in architecture buckets):")
for label, patterns in BEST_ARCHITECTURE_VARIANTS.items():
    for name in sorted(df["source_model"].unique()):
        if any(p.lower() in name.lower() for p in patterns):
            arch = assign_axis_group_key(name, arch_best_spec)
            print(f"  {label:16s} <- {name}  (in {arch})")

plot_model_attack_effectiveness_matrix(
    df,
    out_dir=str(OUT),
    cell_fn=CELL_FN if GROUP_AGG != "auc" else make_metric_cell_fn(METRIC, style="auc"),
    metric=METRIC,
    group_agg=GROUP_AGG,
    group_rows=arch_best_spec,
    group_cols=arch_best_spec,
    block_borders_by="family",
    fname=f"attack_matrix_by_arch_with_best_{METRIC}_{GROUP_AGG}.png",
)
print(f"Saved attack_matrix_by_arch_with_best_{METRIC}_{GROUP_AGG}.png")

Axis order (rows/cols): ['Classical/CNN', 'Classical/CNN*', 'Classical/MLP', 'Classical/MLP*', 'Classical/MLPMixer', 'Classical/ResNet18', 'Hybrid/HPVQL', 'Hybrid/HQE', 'Hybrid/HQuanv', 'Hybrid/HVQL', 'Hybrid/HVQM', 'Quantum/VQM', 'Quantum/VQM*']
Best-variant models (also counted in architecture buckets):
  Classical/CNN*   <- Classical/CNN/conv_channels=[32,64], fc_sizes=[128]  (in Classical/CNN)
  Classical/MLP*   <- Classical/MLP/hidden_sizes=[1024,512,64,64]  (in Classical/MLP)
  Quantum/VQM*     <- Quantum/VQM/ansatz=strong, qdepth=200  (in Quantum/VQM)
Saved attack_matrix_by_arch_with_best_ASR_mean.png


In [ ]:
# 1) Per-model matrix (mean/max uses CELL_FN on METRIC; auc uses metric= directly)
if GROUP_AGG == "auc":
    plot_model_attack_effectiveness_matrix(
        df, out_dir=str(OUT), group_agg="auc", metric=METRIC,
        fname=f"attack_matrix_auc_{METRIC}.png",
    )
else:
    plot_model_attack_effectiveness_matrix(
        df, out_dir=str(OUT), cell_fn=CELL_FN, metric=METRIC, group_agg=GROUP_AGG,
        fname=f"attack_matrix_{METRIC}_{CELL_STYLE}_{GROUP_AGG}.png",
    )

# 2) Structured grouping
GROUP_BY_LEVELS = ["family", "architecture", "variant"]
for level in GROUP_BY_LEVELS:
    plot_model_attack_effectiveness_matrix(
        df, out_dir=str(OUT),
        cell_fn=CELL_FN if GROUP_AGG != "auc" else make_metric_cell_fn(METRIC, style="auc"),
        metric=METRIC,
        group_by=level,
        group_agg=GROUP_AGG,
        fname=f"attack_matrix_by_{level}_{METRIC}_{GROUP_AGG}.png",
    )

In [5]:
# 3) Keyword grouping + filters (custom row/col buckets)

# Columns: Quantum VQM models bucketed by ansatz (rows = all families)
plot_model_attack_effectiveness_matrix(
    df,
    out_dir=str(OUT),
    cell_fn=CELL_FN,
    metric=METRIC,
    group_agg=GROUP_AGG,
    group_rows=MatrixAxisGroups(
        by="keyword",
        keywords=[
            "ansatz=strong",
            "ansatz=basic",
            "ansatz=random",
        ],
        filter_family="Quantum",
        filter_architecture="VQM",
    ),
    group_cols=MatrixAxisGroups(by="architecture"),  # all families as columns
    fname="attack_matrix_cols_quantum_vqm_by_ansatz.png",
)

# Same ansatz columns, but only Classical MLP attackers as rows
plot_model_attack_effectiveness_matrix(
    df,
    out_dir=str(OUT),
    cell_fn=CELL_FN,
    group_rows=MatrixAxisGroups(
        by="keyword",
        keywords=["ansatz=strong", "ansatz=basic", "ansatz=random"],
        filter_family="Quantum",
    ),
    group_cols=MatrixAxisGroups(
        by="architecture",
        filter_family="Classical",
    ),
    fname="attack_matrix_classical_mlp_vs_quantum_ansatz.png",
)
print("Saved keyword/filter examples")

Saved keyword/filter examples


In [4]:
# 4) Inspect how models map into a custom axis spec
from analytics.attacks import parse_canonical_model_name, model_group_key

quantum_ansatz_spec = MatrixAxisGroups(
    by="keyword",
    keywords=["ansatz=strong", "ansatz=basic", "ansatz=random"],
    filter_family="Quantum",
)

print("Quantum models -> ansatz column group:")
for name in sorted(df["source_model"].unique()):
    if "Quantum" not in name:
        continue
    key = assign_axis_group_key(name, quantum_ansatz_spec)
    if key:
        print(f"  {key:16s} <- {name}")

Quantum models -> ansatz column group:
  ansatz=basic     <- Quantum/VQM/ansatz=basic, qdepth=50
  ansatz=random    <- Quantum/VQM/ansatz=random, qdepth=50
  ansatz=strong    <- Quantum/VQM/ansatz=strong, qdepth=100
  ansatz=strong    <- Quantum/VQM/ansatz=strong, qdepth=200
  ansatz=strong    <- Quantum/VQM/ansatz=strong, qdepth=50
